In [ ]:
pip install biopython

In [ ]:
import requests
import pandas as pd
from io import StringIO
import time

BASE = "https://rest.uniprot.org/uniprotkb/search"

def uniprot_search_tsv(query: str, fields: str, size: int = 500, max_rows: int = 5000, polite_sleep: float = 0.2):
    """
    UniProt REST search with pagination using cursor.
    Returns a pandas DataFrame.
    """
    rows = []
    cursor = None
    fetched = 0

    while True:
        params = {
            "query": query,
            "format": "tsv",
            "fields": fields,
            "size": min(size, max_rows - fetched)
        }
        if cursor:
            params["cursor"] = cursor

        r = requests.get(BASE, params=params, timeout=60)
        if r.status_code != 200:
            raise RuntimeError(f"UniProt error {r.status_code}: {r.text[:500]}")

        text = r.text.strip()
        if not text:
            break

        df = pd.read_csv(StringIO(text), sep="\t")
        if df.empty:
            break

        rows.append(df)
        fetched += len(df)

        # stop if reached max_rows
        if fetched >= max_rows:
            break

        # get next cursor from Link header
        link = r.headers.get("Link", "")
        # Link: <...cursor=XXXXX>; rel="next"
        next_cursor = None
        if 'rel="next"' in link:
            # extract cursor=...
            import re
            m = re.search(r'cursor=([^&>]+)', link)
            if m:
                next_cursor = m.group(1)

        if not next_cursor:
            break

        cursor = next_cursor
        time.sleep(polite_sleep)

    if not rows:
        return pd.DataFrame()
    out = pd.concat(rows, ignore_index=True)
    return out

In [ ]:
import numpy as np
import re

# Koliko pozitivnih uzoraka želiš (negativi će se uskladiti na isti broj)
N_POS = 5000          # npr. 2000; možeš povećati na 5000 ako treba
RANDOM_SEED = 42

# Ako želiš samo čovjeka (brže i “čistije”):
ONLY_HUMAN = False    # True -> organism_id:9606

# Feature polja koja dohvaćamo:
FIELDS = "accession,protein_name,organism_name,length,sequence,ft_signal"

In [ ]:
# Queryji
q_org = " AND organism_id:9606" if ONLY_HUMAN else ""

q_pos = f"reviewed:true{q_org} AND ft_signal:*"
q_neg = f"reviewed:true{q_org} AND NOT ft_signal:*"

print("POS query:", q_pos)
print("NEG query:", q_neg)

# Skini malo više negativnih da imaš od čega balansirati
df_pos = uniprot_search_tsv(q_pos, FIELDS, size=500, max_rows=N_POS*2)
df_neg = uniprot_search_tsv(q_neg, FIELDS, size=500, max_rows=N_POS*6)

print("Downloaded POS:", len(df_pos))
print("Downloaded NEG:", len(df_neg))

df_pos.head()

In [ ]:
def clean_protein_seq(seq: str):
    if pd.isna(seq):
        return None
    s = str(seq).strip().upper().replace("*", "")
    # dopusti X (nepoznata aminokiselina), ali izbaci sve ostalo
    if not re.fullmatch(r"[ACDEFGHIKLMNPQRSTVWYX]+", s):
        return None
    return s

def parse_signal_range(ft_signal_field):
    """
    Pokušaj izvući 1..23 iz ft_signal.
    UniProt može vratiti različite formate; ovo hvata najčešći '1..23'.
    """
    if pd.isna(ft_signal_field):
        return (None, None)
    m = re.search(r"(\d+)\.\.(\d+)", str(ft_signal_field))
    if m:
        return (int(m.group(1)), int(m.group(2)))
    return (None, None)

def prep(df: pd.DataFrame, y: int):
    out = df.copy()
    out["sequence"] = out["Sequence"].apply(clean_protein_seq) if "Sequence" in out.columns else out["sequence"].apply(clean_protein_seq)
    out = out.dropna(subset=["sequence"]).reset_index(drop=True)

    # u TSV-u UniProt često kapitalizira nazive stupaca:
    # "Entry" / "Sequence" / "Length" / "Organism" / "Signal peptide" ...
    colmap = {c.lower(): c for c in out.columns}
    def get_col(name_lower):
        return colmap.get(name_lower)

    entry_col = get_col("entry") or get_col("accession")
    org_col   = get_col("organism") or get_col("organism name") or get_col("organism_name")
    len_col   = get_col("length")
    ft_col    = get_col("signal peptide") or get_col("ft_signal")

    out = out.rename(columns={
        entry_col: "accession",
        org_col: "organism",
        len_col: "length",
        ft_col: "ft_signal"
    })

    out["y"] = int(y)

    # signal range (ako postoji)
    out["sp_start"], out["sp_end"] = zip(*out["ft_signal"].apply(parse_signal_range))
    out["sp_len"] = np.where(out["sp_start"].notna() & out["sp_end"].notna(),
                             out["sp_end"] - out["sp_start"] + 1,
                             np.nan)
    return out[["accession","organism","length","sequence","y","sp_start","sp_end","sp_len","ft_signal"]]

df_pos2 = prep(df_pos, 1)
df_neg2 = prep(df_neg, 0)

print("After cleaning POS:", len(df_pos2), "NEG:", len(df_neg2))
df_pos2.head()

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)

# Ako nema dovoljno pozitivnih, smanji N_POS na dostupno
n_pos_avail = len(df_pos2)
n_pos = min(N_POS, n_pos_avail)

df_pos_bal = df_pos2.sample(n=n_pos, random_state=RANDOM_SEED)

# Negativi na isti broj
if len(df_neg2) < n_pos:
    raise RuntimeError(f"Nema dovoljno negativnih ({len(df_neg2)}) za balans {n_pos}. Povećaj max_rows za NEG ili isključi ONLY_HUMAN.")

df_neg_bal = df_neg2.sample(n=n_pos, random_state=RANDOM_SEED)

dataset = pd.concat([df_pos_bal, df_neg_bal], ignore_index=True).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

print("Balanced dataset size:", len(dataset))
print(dataset["y"].value_counts())

dataset.head()

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    dataset,
    test_size=0.2,
    stratify=dataset["y"],
    random_state=RANDOM_SEED
)

train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

train_df.to_csv("train.csv", index=False)
test_df.to_csv("test.csv", index=False)
dataset.to_csv("ml_dataset.csv", index=False)

print("Saved: ml_dataset.csv, train.csv, test.csv")
print("Train size:", len(train_df), "Test size:", len(test_df))

In [ ]:
def write_fasta(df: pd.DataFrame, path: str):
    with open(path, "w") as f:
        for _, r in df.iterrows():
            y = int(r["y"])
            acc = r["accession"]
            sp = ""
            if pd.notna(r["sp_start"]) and pd.notna(r["sp_end"]):
                sp = f"|SP:{int(r['sp_start'])}..{int(r['sp_end'])}"
            header = f">{acc}|y={y}{sp}"
            f.write(header + "\n")
            f.write(r["sequence"] + "\n")

write_fasta(train_df, "train.fasta")
write_fasta(test_df, "test.fasta")

print("Saved: train.fasta, test.fasta")
print("Example header:", open("train.fasta").readline().strip())

In [ ]:
print("Length stats (train):")
print(train_df["length"].describe())

print("\nHow many POS have parsed SP range?")
pos_with_range = train_df[(train_df["y"]==1) & train_df["sp_start"].notna()]
print(len(pos_with_range), "out of", (train_df["y"]==1).sum())

print("\nSP length stats (parsed only):")
print(pos_with_range["sp_len"].describe())

In [ ]:
print("Length stats (train):")
print(train_df["length"].describe())

print("\nHow many POS have parsed SP range?")
pos_with_range = train_df[(train_df["y"]==1) & train_df["sp_start"].notna()]
print(len(pos_with_range), "out of", (train_df["y"]==1).sum())

print("\nSP length stats (parsed only):")
print(pos_with_range["sp_len"].describe())